In [2]:
from snappark import ParkingPredictor

# 1. Load the model
predictor = ParkingPredictor("segformer-epoch=11-val_loss=0.17.ckpt")

# 2. Predict (accepts image path or PIL Image)
mask = predictor.predict("clean_image2.jpg")

# 3. Save or Show
mask.save("pred_parking_mask2.jpg")
mask.show()

Loading model from segformer-epoch=11-val_loss=0.17.ckpt to cpu...


Loading weights: 100%|██████████| 192/192 [00:00<00:00, 1004.25it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]           
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.linear_fuse.weight           

Model loaded successfully.


In [4]:
# Predict on a batch of images
input_image_locations = "batch_images_input"
output_image_locations = "batch_images_output"

# Create a list of input image paths
# recursively search for all jpg images in the input directory and add them to the list
import os


def batch_predict(input_image_locations, output_image_locations):
    list_of_input_images = []

    for root, dirs, files in os.walk(input_image_locations):
        for file in files:
            if file.endswith(".jpg"):
                list_of_input_images.append(os.path.join(root, file))

    for i, image_path in enumerate(list_of_input_images):
        mask = predictor.predict(image_path)
        filename = os.path.basename(image_path)
        filename_without_ext = os.path.splitext(filename)[0]
        mask.save(f"{output_image_locations}/{filename_without_ext}_mask_{i}.jpg")
        mask.show()

In [5]:
batch_predict("batch_images_input", "batch_images_output")

In [10]:
import cv2
import numpy as np
import os
from ultralytics import YOLO

# Load YOLOv8 model (nano version for speed)
yolo_model = YOLO("yolov8n.pt")


def calculate_spots_in_section(section_area, car_area, method="linear"):
    """
    Calculate the number of spots based on section area and reference car area.
    You can modify or add new equations here.
    """
    if car_area <= 0:
        return 0

    if method == "linear":
        # Assume a parking spot is roughly 1.2x the size of a car
        spot_area = car_area * 1.2
        return max(1, int(round(section_area / spot_area)))
    elif method == "conservative":
        # Assume more space between cars (1.5x)
        spot_area = car_area * 1.5
        return max(1, int(round(section_area / spot_area)))
    elif method == "aggressive":
        # Assume tight packing (1.0x)
        spot_area = car_area * 1.0
        return max(1, int(round(section_area / spot_area)))
    else:
        return max(1, int(round(section_area / car_area)))


def superimpose_and_count(
    input_dir,
    mask_dir,
    output_dir,
    min_contour_area=500,
    default_car_area=2500,
    calc_method="linear",
):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if not file.endswith(".jpg"):
                continue

            img_path = os.path.join(root, file)
            filename_without_ext = os.path.splitext(file)[0]

            # Find the corresponding mask file generated by the previous step
            mask_file = None
            for m_file in os.listdir(mask_dir):
                if m_file.startswith(filename_without_ext + "_mask"):
                    mask_file = os.path.join(mask_dir, m_file)
                    break

            if not mask_file:
                print(f"No mask found for {file}")
                continue

            # Read original image and mask
            img = cv2.imread(img_path)
            mask = cv2.imread(mask_file, cv2.IMREAD_GRAYSCALE)

            if img is None or mask is None:
                continue

            # --- YOLOv8 Car Detection for Size Estimation ---
            results = yolo_model(img, verbose=False)
            car_areas = []

            # Class 2 in COCO dataset is 'car'
            for box in results[0].boxes:
                if int(box.cls[0]) == 2:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    area = (x2 - x1) * (y2 - y1)
                    car_areas.append(area)

            # Use median car area to avoid outliers (e.g., cars too close or too far)
            if car_areas:
                reference_car_area = np.median(car_areas)
            else:
                reference_car_area = default_car_area
                print(
                    f"No cars detected in {file}, using default car area: {default_car_area}"
                )

            # Resize mask to match image dimensions if they differ
            if img.shape[:2] != mask.shape[:2]:
                mask = cv2.resize(mask, (img.shape[1], img.shape[0]))

            # Threshold the mask to ensure it is strictly binary
            _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

            # Find contours to identify distinct parking sections (blobs)
            contours, _ = cv2.findContours(
                binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )

            # Filter out small noise based on area
            valid_contours = [
                cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area
            ]

            # Create a colored mask for superimposing (Green overlay)
            colored_mask = np.zeros_like(img)
            colored_mask[binary_mask == 255] = [0, 255, 0]

            # Superimpose the mask onto the original image
            superimposed = cv2.addWeighted(img, 0.7, colored_mask, 0.3, 0)

            # Draw red outlines around the detected sections
            cv2.drawContours(superimposed, valid_contours, -1, (0, 0, 255), 2)

            total_spots_all_sections = 0

            # Process each blob section
            for i, cnt in enumerate(valid_contours):
                section_area = cv2.contourArea(cnt)

                # Estimate spots using the modular function
                estimated_spots = calculate_spots_in_section(
                    section_area, reference_car_area, method=calc_method
                )
                total_spots_all_sections += estimated_spots

                # Find center of the blob for labeling
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cX = int(M["m10"] / M["m00"])
                    cY = int(M["m01"] / M["m00"])
                else:
                    cX, cY = 0, 0

                # Label each section (Class/Blob ID and spot count)
                section_label = f"Sec {i+1}: {estimated_spots} spots"

                # Draw background rectangle for text readability
                (text_width, text_height), baseline = cv2.getTextSize(
                    section_label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
                )
                cv2.rectangle(
                    superimposed,
                    (cX - text_width // 2 - 5, cY - text_height - 5),
                    (cX + text_width // 2 + 5, cY + baseline + 5),
                    (0, 0, 0),
                    -1,
                )

                # Draw the label using OpenCV
                cv2.putText(
                    superimposed,
                    section_label,
                    (cX - text_width // 2, cY),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 255),
                    2,
                    cv2.LINE_AA,
                )

            # Add text displaying the total count across all sections
            cv2.putText(
                superimposed,
                f"Total Spaces: {total_spots_all_sections} (Ref Area: {int(reference_car_area)})",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

            # Save the result
            output_path = os.path.join(
                output_dir, f"{filename_without_ext}_superimposed.jpg"
            )
            cv2.imwrite(output_path, superimposed)
            print(
                f"Processed {file}: Counted {total_spots_all_sections} total spaces across {len(valid_contours)} sections."
            )


# Execute the function
superimpose_and_count(
    "batch_images_input",
    "batch_images_output",
    "batch_images_superimposed",
    calc_method="linear",
)

No cars detected in num_10.jpg, using default car area: 2500
Processed num_10.jpg: Counted 51 total spaces across 2 sections.
No cars detected in num_5.jpg, using default car area: 2500
Processed num_5.jpg: Counted 18 total spaces across 2 sections.
No cars detected in num_6.jpg, using default car area: 2500
Processed num_6.jpg: Counted 29 total spaces across 3 sections.
No cars detected in num_7.jpg, using default car area: 2500
Processed num_7.jpg: Counted 38 total spaces across 2 sections.
No cars detected in num_8.jpg, using default car area: 2500
Processed num_8.jpg: Counted 38 total spaces across 2 sections.
No cars detected in num_9.jpg, using default car area: 2500
Processed num_9.jpg: Counted 13 total spaces across 3 sections.
